In [9]:
import pandas as pd
from pathlib import Path
import pyarrow
import shutil

In [4]:


# Загружаем
sessions = pd.read_pickle("../data/raw/sample_sessions.pkl")
hits = pd.read_pickle("../data/raw/sample_hits.pkl")

print("Sessions shape:", sessions.shape)
print("Hits shape:", hits.shape)

# Первичный просмотр
display(sessions.head())
display(hits.head())

# Информация о таблицах
print(sessions.info())
print(hits.info())

# Проверка пропусков
print("Sessions nulls:\n", sessions.isna().sum())
print("Hits nulls:\n", hits.isna().sum())

# Проверка дубликатов
print("Sessions duplicates:", sessions.duplicated().sum())
print("Hits duplicates:", hits.duplicated().sum())


Sessions shape: (5000, 18)
Hits shape: (5000, 11)


,session_id,client_id,visit_date,visit_time,visit_number,utm_source,utm_medium,utm_campaign,utm_adcontent,utm_keyword,device_category,device_os,device_brand,device_model,device_screen_resolution,device_browser,geo_country,geo_city
27342,9175715433554881536.1629213265.1629213265,2136387730.1629203456,2021-08-17,18:00:00,4,kjsLglQLzykiRbcDiGcD,cpc,NaN,NaN,NaN,mobile,None,Apple,NaN,375x667,Safari,Russia,Moscow
255891,1901950595642554427.1637742649.1637742649,442832381.1637742651,2021-11-24,11:30:49,1,vFcAhRxLfOWKhvxjELkx,organic,okTXSMadDkjvntEHzIjp,LLfCasrxQzJIyuldcuWy,aXQzDWsJuGXeBXexNHjc,desktop,Windows,None,NaN,1600x900,Chrome,Russia,Moscow
245084,1853010306578388863.1623070595.1623070595,431437582.1623070591,2021-06-07,15:00:00,1,fDLlAcSmythWSCVMvqvL,(none),LTuZkdKfxRGVceoWkVyg,JNHcPlZPxEMWDnRiyoBf,NaN,mobile,None,Huawei,NaN,360x780,Chrome,Russia,Moscow
1210450,616131906550852701.1624024318.1624024318,143454388.1623157853,2021-06-18,16:00:00,7,fDLlAcSmythWSCVMvqvL,(none),LTuZkdKfxRGVceoWkVyg,JNHcPlZPxEMWDnRiyoBf,NaN,desktop,None,,NaN,1536x864,YaBrowser,Russia,Moscow
250005,1874993886247173316.1640676552.1640676552,436556033.1640676548,2021-12-28,10:29:12,1,ZpYIoDJMcFzVoPFsHGJL,banner,LEoPHuyFvzoNfnzGgfcd,vCIpmpaGBnIQhyYNkXqp,puhZPIYqKXeFPaUviSjo,mobile,Android,Samsung,NaN,385x854,Chrome,Russia,Kazan


,session_id,hit_date,hit_time,hit_number,hit_type,hit_referer,hit_page_path,event_category,event_action,event_label,event_value
12714978,8218356829458802538.1632380984.1632380984,2021-09-23,NaN,8,event,HbolMJUevblAbkHClEQa,sberauto.com/cars/e994838f?rental_page=rental_car,card_web,view_card,KclpemfoHstknWHFiLit,None
3883637,7016053349963667531.1640372298.1640372392,2021-12-24,99646.0,40,event,NaN,sberauto.com/cars/all/land-rover/range-rover/7...,card_web,photos_all,NaN,None
14623507,9026623559506894998.1637130390.1637130390,2021-11-17,472524.0,45,event,NaN,sberauto.com/cars/all/lada-vaz/vesta/2fc745ed?...,card_web,view_new_card,NaN,None
1472297,3926847712280021957.1635879880.1635879880,2021-11-02,146494.0,12,event,NaN,sberauto.com/cars/all/skoda/karoq/94b3d18e?utm...,card_web,view_card,NaN,None
550848,5565833513341862372.1639236068.1639236068,2021-12-11,186170.0,20,event,NaN,sberauto.com/cars?utm_source_initial=sbol&utm_...,search_form,search_color,fyoQEjeMUkXlgHbrpPUi,None


<class 'pandas.core.frame.DataFrame'>
Index: 5000 entries, 27342 to 935128
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   session_id                5000 non-null   object
 1   client_id                 5000 non-null   object
 2   visit_date                5000 non-null   object
 3   visit_time                5000 non-null   object
 4   visit_number              5000 non-null   int64 
 5   utm_source                5000 non-null   object
 6   utm_medium                5000 non-null   object
 7   utm_campaign              4377 non-null   object
 8   utm_adcontent             4033 non-null   object
 9   utm_keyword               2074 non-null   object
 10  device_category           5000 non-null   object
 11  device_os                 2101 non-null   object
 12  device_brand              4679 non-null   object
 13  device_model              51 non-null     object
 14  device_screen_resolutio

In [5]:
# Построение таргета из hits и merge в sessions (без повторной загрузки)
assert "sessions" in globals() and "hits" in globals()

goal_actions = {
    'sub_car_claim_click','sub_car_claim_submit_click','sub_open_dialog_click',
    'sub_custom_question_submit_click','sub_call_number_click','sub_callback_submit_click',
    'sub_submit_success','sub_car_request_submit_click'
}

hits = hits.assign(is_goal=hits["event_action"].isin(goal_actions).astype("int8"))

goal_by_session = (
    hits.groupby("session_id", as_index=False)["is_goal"]
        .max()
        .rename(columns={"is_goal": "target"})
)

sessions_with_target = sessions.merge(goal_by_session, on="session_id", how="left")
sessions_with_target["target"] = sessions_with_target["target"].fillna(0).astype("int8")

print("Target rate:", round(sessions_with_target["target"].mean(), 4))
sessions_with_target.head(3)

Target rate: 0.0


,session_id,client_id,visit_date,visit_time,visit_number,utm_source,utm_medium,utm_campaign,utm_adcontent,utm_keyword,device_category,device_os,device_brand,device_model,device_screen_resolution,device_browser,geo_country,geo_city,target
0,9175715433554881536.1629213265.1629213265,2136387730.1629203456,2021-08-17,18:00:00,4,kjsLglQLzykiRbcDiGcD,cpc,NaN,NaN,NaN,mobile,None,Apple,NaN,375x667,Safari,Russia,Moscow,0
1,1901950595642554427.1637742649.1637742649,442832381.1637742651,2021-11-24,11:30:49,1,vFcAhRxLfOWKhvxjELkx,organic,okTXSMadDkjvntEHzIjp,LLfCasrxQzJIyuldcuWy,aXQzDWsJuGXeBXexNHjc,desktop,Windows,None,NaN,1600x900,Chrome,Russia,Moscow,0
2,1853010306578388863.1623070595.1623070595,431437582.1623070591,2021-06-07,15:00:00,1,fDLlAcSmythWSCVMvqvL,(none),LTuZkdKfxRGVceoWkVyg,JNHcPlZPxEMWDnRiyoBf,NaN,mobile,None,Huawei,NaN,360x780,Chrome,Russia,Moscow,0


In [6]:
def cr_table(df, by):
    g = df.groupby(by, dropna=False)["target"].agg(["count","mean"])\
          .rename(columns={"count":"visits","mean":"CR"})\
          .sort_values("visits", ascending=False)
    return g

cr_medium = cr_table(sessions_with_target, "utm_medium")
cr_source = cr_table(sessions_with_target, "utm_source")
cr_device = cr_table(sessions_with_target, "device_category")

display(cr_medium.head(10), cr_source.head(10), cr_device)

,visits,CR
utm_medium,,
banner,1488,0.0
cpc,1217,0.0
(none),801,0.0
cpm,653,0.0
referral,393,0.0
organic,178,0.0
email,67,0.0
push,58,0.0
smartbanner,24,0.0


,visits,CR
utm_source,,
ZpYIoDJMcFzVoPFsHGJL,1539,0.0
fDLlAcSmythWSCVMvqvL,801,0.0
kjsLglQLzykiRbcDiGcD,762,0.0
MvfHsxITijuriZxsqZqt,490,0.0
BHcvLfOaCWvWTykYqHVe,329,0.0
bByPQxmDaMXgpHeypKSM,254,0.0
QxAxdyPLuQMEcrdZWdWb,133,0.0
aXQzDWsJuGXeBXexNHjc,85,0.0
vFcAhRxLfOWKhvxjELkx,75,0.0


,visits,CR
device_category,,
mobile,3968,0.0
desktop,974,0.0
tablet,58,0.0


In [10]:
# 1) Найти корень проекта: поднимаемся вверх, пока не найдём data/raw
cwd = Path.cwd().resolve()
ROOT = None
for p in [cwd] + list(cwd.parents):
    if (p / "data" / "raw").exists():
        ROOT = p
        break
assert ROOT is not None, f"Не нашёл корень: {cwd}"

# 2) Правильная папка для сохранения
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

# 3) Если файл уже лежит в notebooks/data/processed — переносим
wrong = cwd / "data" / "processed" / "sessions_with_target.parquet"
dst   = PROC / "sessions_with_target.parquet"
if wrong.exists() and not dst.exists():
    shutil.move(str(wrong), str(dst))
    print("Перенёс:", wrong, "→", dst)

# 4) Сохранить заново в правильное место (на случай обновлений)
sessions_with_target.to_parquet(dst, index=False)
print("Сохранил в:", dst)


Перенёс: C:\Users\vdorofeev\PycharmProjects\data_science_final\notebooks\data\processed\sessions_with_target.parquet → C:\Users\vdorofeev\PycharmProjects\data_science_final\data\processed\sessions_with_target.parquet
Сохранил в: C:\Users\vdorofeev\PycharmProjects\data_science_final\data\processed\sessions_with_target.parquet
